# SETUP DE AMBIENTE E INSTALAÇÃO DE BIBLIOTECAS

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Montagem do Google Drive

In [2]:
# Projeto do TechChallenger da Fase 3
# Assistente médico para pacientes e médicos
# Consulta em modelo fine-tunado baseado em Llama com acrescimo de dados sintéticos produzidos pelo próprio projeto
# Execução de pipeline de tratamento dos dados sinstéticos usando Langchain
# Consulta ao LLM fine-tunado usando LangChain
# Consulta de dados armazenados nos documentos utilizando RAG
# Detecção de execução de funcionalidades orquestradas por LangGraph
# Loging de toda execução

In [3]:
# ========== PRÉ-INSTALAÇÃO (executar uma vez no Google Colab) ==========
!pip install -q requests==2.32.4
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q langchain langgraph langchain-community langchain-huggingface
!pip install -q sentence-transformers faiss-cpu
!pip install -q pandas torch
print("Instalação concluída.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Instalação concluída.


In [4]:
import requests
from langgraph.graph import StateGraph # O import correto do LangGraph

print(f"✅ Versão do requests: {requests.__version__}")

try:
    # Testando os dois pilares que você instalou
    import langchain_core
    import langgraph
    print("✅ LangChain e LangGraph carregados com sucesso!")
except ImportError as e:
    print(f"❌ Erro ao carregar: {e}")

✅ Versão do requests: 2.32.5
✅ LangChain e LangGraph carregados com sucesso!


In [5]:
import requests
import langchain_community
import torch

print(f"✅ Requests version: {requests.__version__}")
print(f"✅ LangChain Community carregada!")
print(f"✅ GPU disponível: {torch.cuda.is_available()}")

✅ Requests version: 2.32.5
✅ LangChain Community carregada!
✅ GPU disponível: True


In [6]:
!pip install -U langchain

In [7]:
# LangChain e Vetores
!pip install -U langchain langchain-community langchain-huggingface langgraph faiss-cpu

# Hugging Face e Fine-tuning
!pip install -U transformers datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [8]:
!pip install -U langchain-text-splitters

In [9]:
# Em vez de: from langchain.text_splitter import ...
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    print("Sucesso com o novo import!")
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    print("Sucesso com o import antigo!")

Sucesso com o novo import!


In [11]:
!pip install -U langchain-text-splitters langchain-community

In [12]:
# Novo padrão de importação
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
# Garante todas as dependências do seu script de uma vez
!pip install -U langchain-text-splitters langchain-community langgraph faiss-cpu langchain-huggingface

In [14]:
!pip install -U langchain-text-splitters langchain-community

In [15]:
!pip install -U langchain-text-splitters

In [16]:
# Tente este novo caminho, que é o padrão atual:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Sucesso!")

Sucesso!


In [17]:
import langchain
print(langchain.__version__)
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Sucesso!")

1.2.12
Sucesso!


In [18]:
import os
import json
import re
import csv
from datetime import datetime
from pathlib import Path

# LangChain / RAG / Fine-tuning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Hugging Face (fine-tuning)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Diretório base para o projeto (usado para dados sintéticos locais, logs, etc.)
BASE_DIR = Path("/content") if os.path.exists("/content") else Path(".")

# Caminho para o Google Drive para modelos e vectorstore
GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL = Path("/content/drive/MyDrive/fine-tuning/fine_tuned_model")

# Caminho para os dados sintéticos no Google Drive
SYNTHETIC_DATA_DRIVE_PATH = Path("/content/drive/MyDrive/fine-tuning/dados_sinteticos")

# Diretórios para dados e modelos
DADOS_DIR = BASE_DIR / "dados_sinteticos_processados"
MODELO_DIR = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL # Modelo fine-tuned será salvo/carregado do Drive
VECTORSTORE_PATH = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL / "faiss_index" # Vectorstore será salvo/carregado do Drive

LOG_OPERACOES = BASE_DIR / "logging-operações.csv"
LOG_CONSULTAS = BASE_DIR / "logging-consultas.csv"

# Cria os diretórios necessários, incluindo os do Google Drive
for d in [DADOS_DIR, MODELO_DIR, VECTORSTORE_PATH, SYNTHETIC_DATA_DRIVE_PATH]:
    d.mkdir(parents=True, exist_ok=True)
print("Imports e configuração OK.")

Imports e configuração OK.


In [19]:
import os
import json
import re
import csv
from datetime import datetime
from pathlib import Path

# LangChain / RAG / Fine-tuning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Hugging Face (fine-tuning)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Diretório base para o projeto (usado para dados sintéticos locais, logs, etc.)
BASE_DIR = Path("/content") if os.path.exists("/content") else Path(".")

# Caminho para o Google Drive para modelos e vectorstore
GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL = Path("/content/drive/MyDrive/fine-tuning/fine_tuned_model")

# Caminho para os dados sintéticos no Google Drive
SYNTHETIC_DATA_DRIVE_PATH = Path("/content/drive/MyDrive/fine-tuning/dados_sinteticos")

# Diretórios para dados e modelos
DADOS_DIR = BASE_DIR / "dados_sinteticos_processados"
MODELO_DIR = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL # Modelo fine-tuned será salvo/carregado do Drive
VECTORSTORE_PATH = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL / "faiss_index" # Vectorstore será salvo/carregado do Drive

LOG_OPERACOES = BASE_DIR / "logging-operações.csv"
LOG_CONSULTAS = BASE_DIR / "logging-consultas.csv"

# Cria os diretórios necessários, incluindo os do Google Drive
for d in [DADOS_DIR, MODELO_DIR, VECTORSTORE_PATH, SYNTHETIC_DATA_DRIVE_PATH]:
    d.mkdir(parents=True, exist_ok=True)
print("Imports e configuração OK.")

Imports e configuração OK.


In [20]:
import os
import json
import re
import csv
from datetime import datetime
from pathlib import Path

# LangChain / RAG / Fine-tuning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Hugging Face (fine-tuning)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Diretório base para o projeto (usado para dados sintéticos locais, logs, etc.)
BASE_DIR = Path("/content") if os.path.exists("/content") else Path(".")

# Caminho para o Google Drive para modelos e vectorstore
GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL = Path("/content/drive/MyDrive/fine-tuning/fine_tuned_model")

# Caminho para os dados sintéticos no Google Drive
SYNTHETIC_DATA_DRIVE_PATH = Path("/content/drive/MyDrive/fine-tuning/dados_sinteticos")

# Diretórios para dados e modelos
DADOS_DIR = BASE_DIR / "dados_sinteticos_processados"
MODELO_DIR = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL # Modelo fine-tuned será salvo/carregado do Drive
VECTORSTORE_PATH = GOOGLE_DRIVE_PATH_FINE_TUNED_MODEL / "faiss_index" # Vectorstore será salvo/carregado do Drive

LOG_OPERACOES = BASE_DIR / "logging-operações.csv"
LOG_CONSULTAS = BASE_DIR / "logging-consultas.csv"

# Cria os diretórios necessários, incluindo os do Google Drive
for d in [DADOS_DIR, MODELO_DIR, VECTORSTORE_PATH, SYNTHETIC_DATA_DRIVE_PATH]:
    d.mkdir(parents=True, exist_ok=True)
print("Imports e configuração OK.")

Imports e configuração OK.


# PIPELINE DE DADOS

## Etapa 1 – Produção e carregamento de dados sintéticos

Os arquivos com prefixo `dados-sinteticos-` já foram criados no projeto. Esta célula carrega e expõe os dados para as etapas seguintes.

In [21]:
#Chave de carga de dados sintéticos
# Configurações
LIGAR_DADOS_SINTETICOS = True
# Removido: GOOGLE_DRIVE_PATH = Path("/content/drive/MyDrive/fine-tuned-model") - agora definido em célula anterior

def definir_caminho_valido(base_path_fallback=None):
    """
    Define qual caminho utilizar: Prioriza o Google Drive (para dados sintéticos),
    caso contrário usa o fallback (local).
    """
    # Usar SYNTHETIC_DATA_DRIVE_PATH para dados sintéticos
    if 'SYNTHETIC_DATA_DRIVE_PATH' in globals() and SYNTHETIC_DATA_DRIVE_PATH.exists():
        print(f"--- Utilizando Google Drive (dados sintéticos): {SYNTHETIC_DATA_DRIVE_PATH} ---")
        return SYNTHETIC_DATA_DRIVE_PATH

    fallback = Path(base_path_fallback) if base_path_fallback else Path(".")
    print(f"--- Google Drive para dados sintéticos não encontrado. Utilizando local: {fallback.absolute()} ---")
    return fallback

def listar_arquivos_dados_sinteticos(caminho_alvo):
    """Lista arquivos com prefixo 'dados-sinteticos-' no diretório escolhido."""
    if not caminho_alvo.exists():
        return []

    arquivos = list(caminho_alvo.glob("dados-sinteticos-*"))
    return [str(f) for f in arquivos]

def carregar_json(path):
    """Carrega um arquivo JSON."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def carregar_todos_dados_sinteticos(lista_arquivos):
    """Carrega os arquivos JSON da lista fornecida em um dicionário."""
    dados = {}
    if not lista_arquivos:
        return dados

    for path in lista_arquivos:
        nome = Path(path).stem
        if path.endswith(".json"):
            try:
                dados[nome] = carregar_json(path)
            except Exception as e:
                dados[nome] = {"erro": str(e)}
    return dados

# --- Execução Principal ---

# 1. Definir o diretório de trabalho (Colab vs Local)
if os.path.exists("/content"):
    os.chdir("/content")
    base_local = Path("/content")
else:
    base_local = Path(".")

dados_brutos = {}

if LIGAR_DADOS_SINTETICOS:
    # 2. Identifica qual caminho está disponível (Drive ou Local) para dados sintéticos
    diretorio_trabalho = definir_caminho_valido(base_local)

    # 3. Lista os arquivos no diretório escolhido
    arquivos_encontrados = listar_arquivos_dados_sinteticos(diretorio_trabalho)

    if arquivos_encontrados:
        print("Arquivos encontrados:", [Path(f).name for f in arquivos_encontrados])
        # 4. Carrega os dados
        dados_brutos = carregar_todos_dados_sinteticos(arquivos_encontrados)

        for k, v in dados_brutos.items():
            print(f"  {k}: {len(v) if isinstance(v, list) else 'dict'} itens")
    else:
        print("Nenhum arquivo 'dados-sinteticos-*' encontrado em nenhum dos locais.")

--- Utilizando Google Drive (dados sintéticos): /content/drive/MyDrive/fine-tuning/dados_sinteticos ---
Arquivos encontrados: ['dados-sinteticos-pedidos-medicos.json', 'dados-sinteticos-laudos-receitas-procedimentos.json', 'dados-sinteticos-protocolos-medicos.json', 'dados-sinteticos-faq-medica.json', 'dados-sinteticos-perguntas-respostas-medicos.json', 'dados-sinteticos-exames-pacientes.json']
  dados-sinteticos-pedidos-medicos: 7 itens
  dados-sinteticos-laudos-receitas-procedimentos: 11 itens
  dados-sinteticos-protocolos-medicos: 7 itens
  dados-sinteticos-faq-medica: 10 itens
  dados-sinteticos-perguntas-respostas-medicos: 7 itens
  dados-sinteticos-exames-pacientes: 10 itens


## Etapa 2 – Pipeline de anonimização

Substitui nomes de pessoas e dados de identificação (CPF, datas de nascimento, CRM) por identificadores anônimos.

### Anonimização

In [22]:
# Mapeamento nome/identificador -> anônimo (preenchido pelo pipeline)
MAPEAMENTO_ANONIMIZACAO = {}

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def anonimizar_texto(texto):
    if LIGAR_ANONIMIZACAO:
        """Substitui CPF, CRM e nomes próprios por placeholders no texto."""
        if not isinstance(texto, str):
            return texto
        # CPF: xxx.xxx.xxx-xx -> [CPF_ANON]
        texto = re.sub(r"\d{3}\.\d{3}\.\d{3}-\d{2}", "[CPF_ANON]", texto)
        # CRM-XX 123456 -> [CRM_ANON]
        texto = re.sub(r"CRM-[A-Z]{2}\s*\d+", "[CRM_ANON]", texto)
        for nome_real, nome_anon in MAPEAMENTO_ANONIMIZACAO.items():
            texto = re.sub(re.escape(nome_real), nome_anon, texto, flags=re.IGNORECASE)
        return texto

def extrair_nomes_para_anonimizar(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Extrai nomes (paciente, medico) dos dados para criar mapeamento."""
        nomes = set()
        for nome_arq, conteudo in dados_dict.items():
            if isinstance(conteudo, list):
                for item in conteudo:
                    if isinstance(item, dict):
                        for k, v in item.items():
                            if v and isinstance(v, str) and k in ("paciente", "medico", "nome"):
                                nomes.add(v.strip())
            elif isinstance(conteudo, dict):
                for k, v in conteudo.items():
                    if k in ("paciente", "medico", "nome") and isinstance(v, str):
                        nomes.add(v.strip())
        return list(nomes)

def construir_mapeamento_anonimizacao(dados_dict):
    if LIGAR_ANONIMIZACAO:
        """Constrói mapa nome_real -> [TIPO_N] (ex: [PACIENTE_1], [MEDICO_1])."""
        global MAPEAMENTO_ANONIMIZACAO
        MAPEAMENTO_ANONIMIZACAO = {}
        nomes = extrair_nomes_para_anonimizar(dados_dict)
        for i, nome in enumerate(nomes, 1):
            MAPEAMENTO_ANONIMIZACAO[nome] = f"[PESSOA_{i}]"
        return MAPEAMENTO_ANONIMIZACAO

def anonimizar_objeto(obj):
    if LIGAR_ANONIMIZACAO:
        """Aplica anonimização recursiva em dict/list/str."""
        if isinstance(obj, str):
            return anonimizar_texto(obj)
        if isinstance(obj, list):
            return [anonimizar_objeto(x) for x in obj]
        if isinstance(obj, dict):
            return {k: anonimizar_objeto(v) for k, v in obj.items()}
        return obj

def executar_pipeline_anonimizacao(dados_brutos, salvar_em=None):
    if LIGAR_ANONIMIZACAO:
        """Pipeline completo: mapeamento + anonimização + opcionalmente salvar."""
        construir_mapeamento_anonimizacao(dados_brutos)
        dados_anon = anonimizar_objeto(dados_brutos)
        if salvar_em:
            Path(salvar_em).mkdir(parents=True, exist_ok=True)
            for nome, conteudo in dados_anon.items():
                path = Path(salvar_em) / f"{nome}.json"
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(conteudo, f, ensure_ascii=False, indent=2)
        return dados_anon

# Executar após carregar dados
if 'dados_brutos' in dir() and dados_brutos:
    if LIGAR_ANONIMIZACAO:
        dados_anonimizados = executar_pipeline_anonimizacao(dados_brutos, DADOS_DIR)
        print("Anonimização concluída. Amostra:", list(dados_anonimizados.keys()))

Anonimização concluída. Amostra: ['dados-sinteticos-pedidos-medicos', 'dados-sinteticos-laudos-receitas-procedimentos', 'dados-sinteticos-protocolos-medicos', 'dados-sinteticos-faq-medica', 'dados-sinteticos-perguntas-respostas-medicos', 'dados-sinteticos-exames-pacientes']


In [23]:
# Chaveamento habilitação de anonimização
# AVALIAR EXCLUSÃO DE BLOCO
LIGAR_ANONIMIZACAO = True

def dados_sinteticos_para_instrucoes(dados_anon):

    if LIGAR_ANONIMIZACAO:
        """Converte dados anonimizados em pares instrução/resposta para fine-tuning."""
        exemplos = []
        for nome_arq, conteudo in dados_anon.items():
            if not isinstance(conteudo, list):
                continue
            for item in conteudo:
                if isinstance(item, dict):
                    if "pergunta" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta"], "response": item["resposta"]})
                    elif "pergunta_medico" in item and "resposta" in item:
                        exemplos.append({"instruction": item["pergunta_medico"], "response": item["resposta"]})
                    elif "protocolo" in item and "descricao" in item:
                        inst = f"Qual o protocolo de {item.get('protocolo', '')}?"
                        exemplos.append({"instruction": inst, "response": item["descricao"]})
                    elif "exame" in item and "interpretacao" in item:
                        inst = f"Interpretar exame: {item.get('exame','')} - Resultado: {item.get('resultado','')}"
                        exemplos.append({"instruction": inst, "response": item["interpretacao"]})
                    elif "tipo" in item and "modelo" in item:
                        exemplos.append({"instruction": f"Modelo de {item['tipo']}", "response": item["modelo"]})
        return exemplos

def formatar_para_llama(instruction, response, template="default"):
    """Formata par instrução/resposta no estilo chat Llama."""
    if template == "default":
        text = f"<s>[INST] {instruction} [/INST] {response} </s>"
    else:
        text = f"Human: {instruction}\nAssistant: {response}"
    return text

def criar_dataset_finetuning(dados_anon, max_amostras=500):
    if LIGAR_ANONIMIZACAO:
        """Cria Dataset Hugging Face para treino."""
        exemplos = dados_sinteticos_para_instrucoes(dados_anon)[:max_amostras]
        if not exemplos:
            # fallback: criar exemplos genéricos a partir de FAQ/protocolos
            for nome_arq, conteudo in dados_anon.items():
                if isinstance(conteudo, list):
                    for item in conteudo:
                        if isinstance(item, dict):
                            for k, v in item.items():
                                if k in ("pergunta", "descricao", "resposta") and v:
                                    exemplos.append({"instruction": "Pergunta sobre saúde", "response": str(v)[:500]})
                    if len(exemplos) >= max_amostras:
                        break
        texts = [formatar_para_llama(e["instruction"], e["response"]) for e in exemplos]
        return Dataset.from_dict({"text": texts})

# CARGA DE MODELO E FINE-TUNING

## Etapa 3 – Carga do modelo base [TinyLlama/TinyLlama-1.1B-Chat-v1.0] do HuggingFace

Preparação dos dados no formato de instrução, fine-tuning com LangChain/Hugging Face (PEFT/LoRA) e salvamento do modelo customizado.

In [24]:
import os
from google.colab import userdata

# Modelo base open source Llama (TinyLlama para caber no Colab; pode trocar por meta-llama/Llama-2-7b-hf com mais RAM)
MODELO_BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Chaveamento habilitação de anonimização
LIGAR_ANONIMIZACAO = True

def carregar_modelo_e_tokenizer(model_name=MODELO_BASE, use_4bit=True):
    """Carrega modelo e tokenizer para fine-tuning com quantização opcional."""
    hf_token = userdata.get('HF_TOKEN') # Pega o token armazenado nos Secrets do Google Colab

    bnb_config = BitsAndBytesConfig(load_in_4bit=use_4bit, bnb_4bit_compute_dtype="float16", bnb_4bit_quant_type="nf4") if use_4bit else None
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=hf_token)
    # Adiciona pad_token se não estiver definido para garantir que o tokenizador funcione corretamente com padding
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", trust_remote_code=True, token=hf_token)
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    return model, tokenizer

def executar_finetuning(dados_anon, modelo_base=MODELO_BASE, output_dir=None, num_epochs=2, batch_size=2):
    if LIGAR_ANONIMIZACAO:
        """Executa fine-tuning (LoRA) e salva o modelo customizado."""
        output_dir = output_dir or str(MODELO_DIR)
        ds = criar_dataset_finetuning(dados_anon)
        model, tokenizer = carregar_modelo_e_tokenizer(modelo_base)

        lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
        model = get_peft_model(model, lora_config)

    def tokenize(examples):
        tokenized_inputs = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
        tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy() # Adiciona labels
        return tokenized_inputs

    ds_tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        logging_steps=10,
        fp16=True,
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=ds_tokenized)
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

# Descomente para rodar (demora e exige GPU no Colab):
if 'dados_anonimizados' in dir():
    caminho_modelo = executar_finetuning(dados_anonimizados, output_dir=str(MODELO_DIR))
    print("Modelo salvo em:", caminho_modelo)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,10.504809


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Modelo salvo em: /content/drive/MyDrive/fine-tuning/fine_tuned_model


# PREPARAÇÃO CHATGPT - Integração com o ChatGPT para análise de resultado dos exames

In [25]:
import os
import openai
from google.colab import userdata

def call_chatgpt_api(system_message: str, user_prompt_base: str, user_value: str) -> str:
    """
    Chama a API do ChatGPT com as mensagens de sistema e usuário fornecidas.

    Args:
        system_message (str): A mensagem de sistema para o modelo.
        user_prompt_base (str): A parte inicial da mensagem do usuário.
        user_value (str): O valor específico fornecido pelo usuário.

    Returns:
        str: A resposta do modelo ChatGPT-4, ou uma mensagem de erro.
    """
    chatgpt_api_key = userdata.get('FIAP-TechChallenger-Fase3-ChatGPT-API-KEY')

    if not chatgpt_api_key:
        return "Erro: A chave 'FIAP-TechChallenger-Fase3-ChatGPT-API-KEY' não foi encontrada nos secrets do Colab."

    client = openai.OpenAI(api_key=chatgpt_api_key)

    user_message_full = f"{user_prompt_base} {user_value}"

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message_full}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content

    except openai.APIError as e:
        return f"Erro da API do OpenAI: {e}"
    except Exception as e:
        return f"Ocorreu um erro inesperado ao chamar a API do ChatGPT: {e}"

print("Função 'call_chatgpt_api' criada com sucesso.")


Função 'call_chatgpt_api' criada com sucesso.


### Teste da função `call_chatgpt_api` de integração com o ChatGPT

In [26]:
# Exemplo de uso da função:
system_msg = "Como especialista em exames clínicos de um hospital, analise o resultado de exame informado:"
user_prompt = "O resultado do exame foi:"
user_val = "Cálculo renal de 5mm no rim direito"

print(f"Enviando prompt para o modelo GPT-4:\nSistema: {system_msg}\nUsuário: {user_prompt} {user_val}")
chatgpt_response = call_chatgpt_api(system_msg, user_prompt, user_val)

print("\n--- Resposta do ChatGPT-4 (via função) ---")
print(chatgpt_response)
print("------------------------------------------")


Enviando prompt para o modelo GPT-4:
Sistema: Como especialista em exames clínicos de um hospital, analise o resultado de exame informado:
Usuário: O resultado do exame foi: Cálculo renal de 5mm no rim direito

--- Resposta do ChatGPT-4 (via função) ---
O resultado indica que o paciente tem um cálculo renal (pedra no rim) de 5mm no rim direito. O tamanho de 5mm é considerado pequeno e, em muitos casos, pode passar pelo trato urinário sem necessidade de intervenção cirúrgica. No entanto, isso pode causar desconforto e dor significativos, dependendo da localização exacta da pedra. 

A dor geralmente ocorre quando a pedra começa a se mover e a irritar o revestimento do ureter. Sintomas comuns incluem dor nas costas ou lado, dor abdominal, sangue na urina, náuseas e vômitos.

Para tratamento, deve-se aumentar a ingestão de líquidos para ajudar a pedra a passar. Em alguns casos, pode ser necessária medicação para aliviar a dor. Se a pedra não passar por conta própria, ou se causar outros pr

# VECTOR_STORE + RAG + LANGRAPH + ASSISTENTE

## Etapa 4 e 5 – Assistente com RAG e LangGraph

Construção do índice vetorial (RAG), grafo LangGraph para roteamento de intenções (sugerir tratamentos, análise de exames, alertas de risco, dúvidas gerais) e assistente com logs e regras de segurança.

In [27]:
# RAG - preparação da funções

def documentos_para_rag(dados_anon):
    """
    Converte dados anonimizados em uma lista de objetos Document para RAG.

    Args:
        dados_anon (dict): Dicionário com nome do arquivo e seu conteúdo.
    Returns:
        list: Lista de objetos Document.
    """
    docs = []

    for nome_arq, conteudo in dados_anon.items():
        # Caso o conteúdo seja uma lista de itens (ex: linhas de uma tabela)
        if isinstance(conteudo, list):
            for i, item in enumerate(conteudo):
                if isinstance(item, dict):
                    texto = json.dumps(item, ensure_ascii=False)
                    docs.append(Document(
                        page_content=texto,
                        metadata={"fonte": nome_arq, "indice": i}
                    ))

        # Caso o conteúdo seja um dicionário único
        elif isinstance(conteudo, dict):
            texto = json.dumps(conteudo, ensure_ascii=False)
            docs.append(Document(
                page_content=texto,
                metadata={"fonte": nome_arq}
            ))

    return docs

def construir_vectorstore(dados_anon, persist_path=None):

    """Cria FAISS a partir dos documentos e opcionalmente persiste."""
    docs = documentos_para_rag(dados_anon)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
    splits = splitter.split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    vectorstore = FAISS.from_documents(splits, embeddings)
    if persist_path:
       vectorstore.save_local(str(persist_path))
       return vectorstore

def log_operacao(operacao, detalhe="", arquivo=LOG_OPERACOES):
    """Registra operação em logging-operações.csv com data e hora (minuto e segundo)."""
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow(["data", "hora", "minuto", "segundo", "operacao", "detalhe"])
        now = datetime.now()
        w.writerow([now.strftime("%Y-%m-%d"), now.strftime("%H"), now.strftime("%M"), now.strftime("%S"), operacao, detalhe])

def log_consulta(tipo_usuario, nome_usuario, pergunta, resposta_assistente, arquivo=LOG_CONSULTAS):
    """Registra consulta em logging-consultas.csv."""
    with open(arquivo, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if f.tell() == 0:
            w.writerow(["data", "hora", "minuto", "segundo", "tipo_usuario", "nome_usuario", "pergunta", "resposta_assistente"])
        now = datetime.now()
        w.writerow([now.strftime("%Y-%m-%d"), now.strftime("%H:%M:%S"), now.strftime("%M"), now.strftime("%S"), tipo_usuario, nome_usuario, pergunta[:500], resposta_assistente[:1000]])

### Construção do VectorStore

In [28]:
# Construir o VectorStore para RAG
# Garante que dados_anonimizados (que agora são os dados brutos) estão sendo usados

print(f'VECTORSTORE_PATH: {VECTORSTORE_PATH}')

if 'dados_anonimizados' in dir() and dados_anonimizados:
    print("Construindo o VectorStore...")
    vectorstore = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)
    print(f"VectorStore construído e salvo em: {VECTORSTORE_PATH}")
else:
    print("Dados não disponíveis para construir o VectorStore. Certifique-se de que 'dados_brutos' foram carregados e 'dados_anonimizados' foram definidos.")

VECTORSTORE_PATH: /content/drive/MyDrive/fine-tuning/fine_tuned_model/faiss_index
Construindo o VectorStore...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VectorStore construído e salvo em: /content/drive/MyDrive/fine-tuning/fine_tuned_model/faiss_index


In [29]:
# LangGraph - versão mais atualizada da classe


import json
from collections import defaultdict
import re
from typing import TypedDict, Literal

class EstadoAssistente(TypedDict):
    pergunta: str
    tipo_usuario: Literal["paciente", "medico"]
    nome_usuario: str
    intencao: str  # "tratamentos", "analise_exames", "alertas_risco", "duvida_geral"
    contexto_rag: str
    resposta: str
    fonte: str

def classificar_intencao(pergunta: str) -> str:
    """Classifica a intenção do usuário a partir do texto (simplificado; pode usar LLM depois)."""
    p = pergunta.lower()
    if any(x in p for x in ["tratamento", "procedimento", "como tratar", "terapia", "protocolo"]):
        return "tratamentos"
    if any(x in p for x in ["exame", "resultado", "laudo", "hemograma", "glicemia", "creatinina", "interpretar"]):
        return "analise_exames"
    if any(x in p for x in ["risco", "grave", "alerta", "urgente", "perigo", "crítico"]):
        return "alertas_risco"
    return "duvida_geral"

def node_classificar(state: EstadoAssistente) -> EstadoAssistente:
    state["intencao"] = classificar_intencao(state["pergunta"])
    log_operacao("classificacao_intencao", state["intencao"])
    return state

def node_buscar_rag(state: EstadoAssistente, vectorstore) -> EstadoAssistente:
    if vectorstore is None:
        state["contexto_rag"] = ""
        state["fonte"] = "modelo"
        return state

    # Aumentar k para garantir que mais documentos sejam recuperados
    docs = vectorstore.similarity_search(state["pergunta"], k=10) # Aumentado de 5 para 10

    # Remover a filtragem agressiva para trazer 'todos os resultados possíveis'
    # O contexto RAG agora conterá todos os documentos encontrados pela busca de similaridade
    state["contexto_rag_docs"] = docs # Armazena os objetos Document para categorização posterior
    state["contexto_rag"] = "\n\n".join(d.page_content for d in docs) # Mantém o string para compatibilidade
    state["fonte"] = "RAG: " + ", ".join(set(d.metadata.get("fonte", "doc") for d in docs)) if docs else "Nenhuma fonte específica encontrada"
    return state

def limpar_jsonl(input_string):
    linhas_limpas = []
    # Divide por quebra de linha
    linhas = input_string.strip().split('\n')

    for linha in linhas:
        # Remove espaços em branco nas extremidades
        linha_formatada = linha.strip()

        if not linha_formatada:
            continue

        try:
            # Valida se é um JSON válido
            objeto_json = json.loads(linha_formatada)
            # Reconverte para string em uma única linha (compacto)
            linhas_limpas.append(json.dumps(objeto_json, ensure_ascii=False))
        except json.JSONDecodeError as e:
            print(f"Erro ao processar linha: {linha_formatada[:30]}... Erro: {e}")

    return "\n".join(linhas_limpas)


def node_gerar_resposta(state: EstadoAssistente, llm_inference_fn) -> EstadoAssistente:

    aviso = " Lembre-se: não prescrevo medicamentos nem substituo o médico. Sempre confirme com seu médico de acompanhamento ou médico de plantão."

    # 1. Obter resposta direta do LLM
    llm_response_text = ""
    if llm_inference_fn:
        # Prompt mais direto para evitar que o LLM repita as instruções
        prompt_llm = f"Como um assistente médico, explique de forma clara e concisa o seguinte: {state['pergunta']}"
        llm_response_text += llm_inference_fn(prompt_llm)
    else:
        llm_response_text += "(Modelo LLM não disponível para resposta direta, ou erro na inferência)."


    # 3. Formatar a saída final no State
    final_response_parts = []
    final_response_parts.append(f"\nOlá {state['tipo_usuario']} {state['nome_usuario']}, eu sou o seu assistente médico.")
    final_response_parts.append(f"\nVocê pesquisou: {state['pergunta']}")
    final_response_parts.append("\n\nEu encontrei os seguintes resultados:\n")

    final_response_parts.append("\n[ LLM - Resultado ]\n")
    final_response_parts.append(llm_response_text)
    #final_response_parts.append("\n[ RAG - Resultados ]\n")


# ------ retorno do RAG -----------------
    # Execução da Limpeza do retorno do RAG
    jsonl_final = limpar_jsonl(state["contexto_rag"])

    with open("dados_saude_limpos.jsonl", "w", encoding="utf-8") as f:
        f.write(jsonl_final)

    # Processando o JSONL para uma lista de dicionários (apenas uma vez)
    # Adicionado tratamento para linhas vazias para evitar erros no json.loads
    dados = [json.loads(linha) for linha in jsonl_final.strip().split('\n') if linha.strip()]

    contexto_rag_json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)

    # Lógica de Categorização
    categorias = defaultdict(list)

    for item in dados:
        if "protocolo" in item:
            categorias["Protocolos"].append(f"{item['protocolo']} ({item.get('instituicao', 'N/A')})")

        elif "paciente" in item:
            categorias["Dados de Pacientes (Exames)"].append(f"Paciente {item['paciente']} - Exame: {item['exame']} | Resultado: {item['resultado']}")

            #-----------------------------------------------------------------------------------------------
            # Início da análise de exame feita pela LLM externo ChatGPT para avaliar o resultado dos exame
            #-----------------------------------------------------------------------------------------------

            system_msg = "Como especialista em exames clínicos de um hospital, analise o resultado de exame informado e informar o nível de gravidade conforme a escala universal Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)"
            user_prompt = "O resultado do exame do paciente " + item['paciente'] + " foi:"
            user_val = item['resultado']

            #print(f"Enviando prompt para o modelo GPT-4:\nSistema: {system_msg}\nUsuário: {user_prompt} {user_val}")
            print(f"\n- Avaliando o resultado do exame do paciente {item['paciente']} com o ChatGPT para obtenção de mais detalhes e classificação conforme a Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)")
            chatgpt_response = call_chatgpt_api(system_msg, user_prompt, user_val)

            print(f"\n- ok - Resposta da análise e classificação do resultado do exame do paciente {item['paciente']} pelo ChatGPT-4 - obtida ---")
            #print(chatgpt_response)
            final_response_parts.append(chatgpt_response)
            print(f"\n- ok - Resposta da análise e classificação do resultado do exame do paciente {item['paciente']} pelo ChatGPT-4 - gravada na saída final ao usuário ---")

            #-----------------------------------------------------------------------------------------------
            # Fim da análise de exame feita pela LLM para avaliar o resultado dos exame
            #-----------------------------------------------------------------------------------------------

        elif "pergunta" in item or "pergunta_medico" in item:
            pergunta = item.get("pergunta") or item.get("pergunta_medico")
            resposta = item.get("resposta", "")[:80]
            categorias["Dúvidas e Orientações (FAQ)"].append(f"{pergunta} Resposta: {resposta}...")

        elif "tipo" in item:
            modelo = item.get("modelo", "")[:100]
            if "Receita" in item["tipo"]:
                categorias["Modelos de Documentos"].append(f"{item['tipo']}: {modelo}...")
            else:
                categorias["Protocolos"].append(f"{item['tipo']} - {modelo}...")

    # Montagem da resposta final (Correção da concatenação)
    for cat, itens in categorias.items():
        # Convertendo len(itens) para str() ou usando f-string
        header = f"\n[ {cat} ] - quantidade : {len(itens)}"
        final_response_parts.append(header)

        for linha in itens:
            final_response_parts.append(f"- {linha}")

    # ------ retorno do RAG -----------------

    state["resposta"] = "\n".join(final_response_parts)
    state["fonte"] = "LLM + " + state.get("fonte", "RAG")

    return state

In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel # Importar PeftModel para carregar adaptadores

def construir_grafo_assistente(vectorstore, llm_inference_fn):
    """Monta o grafo LangGraph: classificar -> buscar RAG -> gerar resposta."""
    def buscar(state):

        return node_buscar_rag(state, vectorstore)

    def gerar(state):

        return node_gerar_resposta(state, llm_inference_fn)

    workflow = StateGraph(EstadoAssistente)
    workflow.add_node("classificar", node_classificar)
    workflow.add_node("buscar_rag", buscar)
    workflow.add_node("gerar_resposta", gerar)
    workflow.set_entry_point("classificar")
    workflow.add_edge("classificar", "buscar_rag")
    workflow.add_edge("buscar_rag", "gerar_resposta")
    workflow.add_edge("gerar_resposta", END)
    return workflow.compile()

def aplicar_regras_seguranca(texto: str) -> str:
    """Sempre responda em língua portuguesa do Brasil e garante que a resposta inclua orientação de não prescrever e confirmar com médico."""
    if "confirmar" not in texto.lower() and "médico" not in texto.lower():
        texto += " Recomendo confirmar as informações com seu médico de acompanhamento ou médico de plantão para os próximos passos."
    if "medicamento" in texto.lower() or "remédio" in texto.lower():
        texto += " Não sugiro medicamentos; oriente-se sempre com o médico que acompanha o caso ou o médico de plantão."
    return texto

def inferencia_llm_simulada(prompt: str, modelo_path=None) -> str:
    """Inferência com modelo fine-tunado (ou simulada se modelo não disponível)."""

    print(f"\n modelo_path considerado para inferência: {modelo_path} \n")

    try:
        # Sempre carrega o modelo base primeiro, com a quantização de 4 bits (default de carregar_modelo_e_tokenizer)
        base_model, tokenizer = carregar_modelo_e_tokenizer(model_name=MODELO_BASE)
        model = base_model # Começa com o modelo base

        # Tenta carregar o adaptador PEFT se o modelo_path for fornecido e existir
        if modelo_path and Path(modelo_path).exists():
            print(f"DEBUG: Tentando carregar adaptador PEFT de {modelo_path}")
            try:
                # Carrega o adaptador LoRA e o mescla ao modelo base para inferência
                model = PeftModel.from_pretrained(base_model, modelo_path)
                model = model.merge_and_unload() # Mescla os pesos LoRA no modelo base
                print(f"DEBUG: Adaptador PEFT carregado e mesclado com sucesso.")
            except Exception as peft_e:
                print(f"DEBUG: Erro ao carregar/mesclar adaptador PEFT de {modelo_path}: {peft_e}")
                print(f"DEBUG: Caindo para inferência com o modelo base (sem fine-tuning).")
                model = base_model # Reverte para o modelo base se houver erro no PEFT
        else:
            print(f"DEBUG: Caminho do modelo fine-tuned ({modelo_path}) não fornecido ou não existe. Usando modelo base para inferência.")
            model = base_model

        if not model or not tokenizer:
            print(f"DEBUG: Falha crítica ao inicializar modelo ou tokenizer para inferência.")
            return f"Resposta baseada no contexto e na intenção. [Falha crítica ao carregar modelo LLM para inferência]. Sempre consulte seu médico para condutas."

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.3)
        return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    except Exception as e:
        print(f"DEBUG: Erro geral durante a inferência do LLM: {e}")
        log_operacao("erro_inferencia", str(e))
        return f"Resposta baseada no contexto e na intenção. [Erro na inferência do LLM: {e}]. Sempre consulte seu médico para condutas."


### Testando o modelo fine-tuned e do Assistente

In [31]:
#Assistente


def executar_assistente(pergunta: str, tipo_usuario: str, nome_usuario: str, grafo) -> str:
    """
    Executa o fluxo do assistente: grafo (classificar -> RAG -> resposta), aplica regras de segurança e loga.
    Nunca prescreve; sempre orienta a confirmar com médico.
    """
    estado_inicial = {
        "pergunta": pergunta,
        "tipo_usuario": tipo_usuario,
        "nome_usuario": nome_usuario,
        "intencao": "",
        "contexto_rag": "",
        "resposta": "",
        "fonte": "",
    }
    resultado = grafo.invoke(estado_inicial)
    resposta = resultado.get("resposta", "")
    resposta = aplicar_regras_seguranca(resposta)
    log_consulta(tipo_usuario, nome_usuario, pergunta, resposta)
    log_operacao("consulta_assistente", f"{tipo_usuario}/{nome_usuario}")
    return resposta

In [33]:
# Teste para o assistente com RAG e, opcionalmente, com o modelo fine-tuned.

if "dados_anonimizados" in dir() and dados_anonimizados:
    print("Preparando para testar o assistente...")

    # Reconstruir o vectorstore para RAG
    # Isso garante que o vectorstore esteja carregado ou atualizado para o teste
    vectorstore_test = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)

    # Definir se usará o modelo fine-tunado (True/False) para este teste
    # Alterado para True para testar o comportamento do LLM e a busca de todos os resultados RAG
    usar_modelo_finetuned_test = True

    llm_fn_test = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned_test else None

    # Construir o grafo do assistente para o teste
    grafo_test = construir_grafo_assistente(vectorstore_test, llm_fn_test)

    # Simular uma consulta
    pergunta_teste = "diabetes"
    tipo_usuario_teste = "Paciente"
    nome_usuario_teste = "João"

    print(f"\nConsultando o assistente (usuário: {nome_usuario_teste}, tipo: {tipo_usuario_teste}, assunto: \"{pergunta_teste}\")...")
    resposta_teste = executar_assistente(pergunta_teste, tipo_usuario_teste, nome_usuario_teste, grafo_test)

    print("\n\n\n\n\n--- Resposta do Assistente Médico ---")
    print(resposta_teste)
    print("-------------------------------------")
    print("\n Teste do assistente concluído.")

else:
    print("Dados anonimizados não disponíveis. Certifique-se de que os dados sintéticos foram carregados e o pipeline de anonimização (mesmo que bypassado) foi executado.")

Preparando para testar o assistente...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Consultando o assistente (usuário: João, tipo: Paciente, assunto: "diabetes")...

 modelo_path considerado para inferência: /content/drive/MyDrive/fine-tuning/fine_tuned_model 



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEBUG: Tentando carregar adaptador PEFT de /content/drive/MyDrive/fine-tuning/fine_tuned_model


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG: Adaptador PEFT carregado e mesclado com sucesso.

- Avaliando o resultado do exame do paciente [PESSOA_3] com o ChatGPT para obtenção de mais detalhes e classificação conforme a Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_3] pelo ChatGPT-4 - obtida ---

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_3] pelo ChatGPT-4 - gravada na saída final ao usuário ---

- Avaliando o resultado do exame do paciente [PESSOA_13] com o ChatGPT para obtenção de mais detalhes e classificação conforme a Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_13] pelo ChatGPT-4 - obtida ---

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_13] pelo ChatGPT-4 - gravada na saída final ao usuário ---

- Avaliando o resultad

# PROGRAMA PRINCIPAL

In [32]:
from IPython.display import clear_output

## PROGRAMA PRINCIPAL ## ASSISTENTE MÉDICO INTERATIVO ## 2026 - TECHCHALLENGER - FASE 3 - FIAP

clear_output(wait=True)
print("Inicializando o assistente médico interativo...")

# Certifique-se de que 'dados_anonimizados' está disponível do pipeline anterior
if "dados_anonimizados" not in dir() or not dados_anonimizados:
    print("Nenhum dado anonimizado carregado. Carregue os dados sintéticos e execute a anonimização antes de rodar o assistente interativo.")
else:
    # Construir o vectorstore para RAG
    # Reconstruímos aqui para garantir que o vectorstore esteja atualizado ou carregado
    vectorstore = construir_vectorstore(dados_anonimizados, persist_path=VECTORSTORE_PATH)

    # Definir se usará o modelo fine-tunado (True/False)
    # Por padrão, manteremos como False, a menos que o fine-tuning tenha sido explicitamente executado.
    usar_modelo_finetuned = True # Altere para True se você executou o fine-tuning
    llm_fn = (lambda p: inferencia_llm_simulada(p, str(MODELO_DIR))) if usar_modelo_finetuned else None

    # Construir o grafo do assistente com o vectorstore e a função de inferência do LLM
    grafo = construir_grafo_assistente(vectorstore, llm_fn)

    print("\n\n Assistente pronto. Digite 'sair' a qualquer momento para encerrar.\n")

    tipo_usuario = input("\n Você é paciente ou médico? (paciente/medico):\n").strip().lower() or "paciente"

    #tipo_usuario = input("\n Você é paciente ou médico? (paciente/medico): ").strip().lower() or "paciente"

    if tipo_usuario not in ("paciente", "medico"):
        tipo_usuario = "paciente"

    nome_usuario = input("\n Qual o seu nome (ou do paciente)?:\n").strip() or "Não informado"
    #nome_usuario = input("Qual o seu nome (ou do paciente)?: ").strip() or "Não informado"
    log_operacao("inicio_sessao", f"{tipo_usuario} - {nome_usuario}")

    while True:
        #pergunta = input("\nSua pergunta: ").strip()
        pergunta = input("\n Assunto:   (digite sair para encerrar) \n").strip().lower() or "paciente"
        if not pergunta or pergunta.lower() == 'sair':
            print("Encerrando o assistente.")
            break

        print("\n\n Processando sua pesquisa...\n\n ")
        resposta = executar_assistente(pergunta, tipo_usuario, nome_usuario, grafo)

        print("\n\n--- Assistente Médico --- \n")
        print(resposta)
        print("------------------------- \n\n")

    log_operacao("fim_sessao", nome_usuario)
    print("Sessão encerrada. Logs em logging-operações.csv e logging-consultas.csv.")

Inicializando o assistente médico interativo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




 Assistente pronto. Digite 'sair' a qualquer momento para encerrar.


 Você é paciente ou médico? (paciente/medico):
medico

 Qual o seu nome (ou do paciente)?:
dr. Bruno

 Assunto:   (digite sair para encerrar) 
obesidade


 Processando sua pesquisa...

 

 modelo_path considerado para inferência: /content/drive/MyDrive/fine-tuning/fine_tuned_model 



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

DEBUG: Tentando carregar adaptador PEFT de /content/drive/MyDrive/fine-tuning/fine_tuned_model


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG: Adaptador PEFT carregado e mesclado com sucesso.

- Avaliando o resultado do exame do paciente [PESSOA_3] com o ChatGPT para obtenção de mais detalhes e classificação conforme a Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_3] pelo ChatGPT-4 - obtida ---

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_3] pelo ChatGPT-4 - gravada na saída final ao usuário ---

- Avaliando o resultado do exame do paciente [PESSOA_13] com o ChatGPT para obtenção de mais detalhes e classificação conforme a Escala CTCAE (Critérios de Terminologia Comum para Eventos Adversos)

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_13] pelo ChatGPT-4 - obtida ---

- ok - Resposta da análise e classificação do resultado do exame do paciente [PESSOA_13] pelo ChatGPT-4 - gravada na saída final ao usuário ---


--- Assistente Médico